In [1]:
import pandas as pd
import re
import html
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
tqdm.pandas()

print("Library berhasil dimuat")

Library berhasil dimuat


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv('rawData/dataNews.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         234 non-null    object
 1   publish_date  234 non-null    object
 2   content       234 non-null    object
dtypes: object(3)
memory usage: 5.6+ KB


In [3]:
initial_len = len(df)
df.drop_duplicates(subset=['content'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total berita: {len(df)} (dihapus {initial_len - len(df)} duplikat)")

Total berita: 234 (dihapus 0 duplikat)


In [4]:
def clean_news_bertopic(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities (mis. &amp; → &)
    text = html.unescape(text)

    # Hapus elemen non-konten — pola multiline (header, byline, artefak web)
    non_content_patterns = [
        r'(?i)^baca juga[:\-\s].*$',
        r'(?i)^simak juga[:\-\s].*$',
        r'(?i)^lihat juga[:\-\s].*$',
        r'(?i)^advertisement.*$',
        r'(?i)^scroll to continue.*$',
        r'(?i)^video[:\-\s].*$',
        r'(?i)^foto[:\-\s].*$',
        r'(?i)^reporter[:\-\s].*$',       # byline reporter
        r'(?i)^editor[:\-\s].*$',         # byline editor
        r'(?i)^penulis[:\-\s].*$',        # byline penulis
        r'(?i)^sumber[:\-\s].*$',         # label sumber berita
    ]
    for pattern in non_content_patterns:
        text = re.sub(pattern, '', text, flags=re.MULTILINE)

    # Hapus label inline artefak web
    text = re.sub(
        r'(?i)SCROLL TO CONTINUE WITH CONTENT|ADVERTISEMENT'
        r'|BACA JUGA:|SIMAK JUGA:|LIHAT JUGA:|VIDEO:|FOTO:'
        r'|REPORTER:|EDITOR:|PENULIS:|SUMBER:',
        '',
        text
    )

    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Normalisasi tanda kutip tipografi → standar
    text = re.sub(r'[""]', '"', text)
    text = re.sub(r"['']", "'", text)

    # Hapus karakter non-teks (tanda baca, simbol, angka)
    text = re.sub(r'[^\w\s]', ' ', text)   # hapus tanda baca & simbol
    text = re.sub(r'\d+', ' ', text)       # hapus angka

    # Bersihkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['data_cleaning'] = df['content'].progress_apply(clean_news_bertopic)
print("Data cleaning selesai")

100%|██████████| 234/234 [00:00<00:00, 1207.80it/s]

Data cleaning selesai


In [5]:
def case_folding(text):
    if not isinstance(text, str):
        return ""
    return text.lower()

df['case_folding'] = df['data_cleaning'].progress_apply(case_folding)
print("Case folding selesai")

100%|██████████| 234/234 [00:00<00:00, 234296.28it/s]

Case folding selesai


In [6]:
def normalize_minimal_news(text):
    normalization_dict = {
        # Singkatan umum dalam teks berita & kutipan
        'yg'    : 'yang',
        'dg'    : 'dengan',
        'dgn'   : 'dengan',
        'tsb'   : 'tersebut',
        'svp'   : 'sebagai',
        'tdk'   : 'tidak',
        'tak'   : 'tidak',
        'krn'   : 'karena',
        'sdh'   : 'sudah',
        'blm'   : 'belum',
        'utk'   : 'untuk',
        'thd'   : 'terhadap',
        'ttg'   : 'tentang',
        'jd'    : 'jadi',
        'pd'    : 'pada',
        'dlm'   : 'dalam',
        'dpt'   : 'dapat',
        'hrs'   : 'harus',
        'msh'   : 'masih',
        'jk'    : 'jika',
        'dr'    : 'dari',
        'kpd'   : 'kepada',
        'spt'   : 'seperti',
        'stlh'  : 'setelah',
        'sblm'  : 'sebelum',
        'org'   : 'orang',
        'pem'   : 'pemerintah',
        'pemda' : 'pemerintah daerah',
        'pemkot': 'pemerintah kota',
        'pemkab': 'pemerintah kabupaten',
    }

    words = text.split()
    normalized = []
    for word in words:
        replacement = normalization_dict.get(word)
        if replacement is None:
            normalized.append(word)         # tidak ada di kamus → pertahankan
        elif replacement:
            normalized.append(replacement)  # ada pengganti → ganti
        # else: pengganti kosong → buang

    return ' '.join(normalized)

df['normalization'] = df['case_folding'].progress_apply(normalize_minimal_news)
print("Normalisasi minimal selesai")
print(f"Ukuran kamus normalisasi: {29} istilah kritis")

100%|██████████| 234/234 [00:00<00:00, 15012.19it/s]

Normalisasi minimal selesai
Ukuran kamus normalisasi: 29 istilah kritis


In [8]:
print("=" * 60)
print("LAPORAN KUALITAS PREPROCESSING — DATA BERITA (BERTopic)")
print("=" * 60)

df['orig_length']  = df['content'].str.len()
df['final_length'] = df['normalization'].str.len()
df['orig_words']   = df['content'].apply(lambda x: len(str(x).split()))
df['final_words']  = df['normalization'].apply(lambda x: len(str(x).split()))

avg_orig    = df['orig_length'].mean()
avg_final   = df['final_length'].mean()
avg_orig_w  = df['orig_words'].mean()
avg_final_w = df['final_words'].mean()

print(f"\nStatistik panjang teks:")
print(f"  Rata-rata panjang asli  : {avg_orig:.1f} karakter")
print(f"  Rata-rata panjang akhir : {avg_final:.1f} karakter")
print(f"  Reduksi                 : {((avg_orig - avg_final) / avg_orig * 100):.1f}%")

print(f"\nStatistik jumlah kata:")
print(f"  Rata-rata kata asli  : {avg_orig_w:.1f}")
print(f"  Rata-rata kata akhir : {avg_final_w:.1f}")
print(f"  Reduksi              : {((avg_orig_w - avg_final_w) / avg_orig_w * 100):.1f}%")

print(f"\nDistribusi panjang teks akhir:")
print(f"  Terpendek : {df['final_length'].min()} karakter")
print(f"  Terpanjang: {df['final_length'].max()} karakter")
print(f"  Median    : {df['final_length'].median():.1f} karakter")

LAPORAN KUALITAS PREPROCESSING — DATA BERITA (BERTopic)

Statistik panjang teks:
  Rata-rata panjang asli  : 3042.1 karakter
  Rata-rata panjang akhir : 2850.5 karakter
  Reduksi                 : 6.3%

Statistik jumlah kata:
  Rata-rata kata asli  : 416.6
  Rata-rata kata akhir : 404.4
  Reduksi              : 2.9%

Distribusi panjang teks akhir:
  Terpendek : 254 karakter
  Terpanjang: 25371 karakter
  Median    : 2309.5 karakter


In [9]:
for i in range(min(2, len(df))):
    print(f"\nBERITA {i + 1}:")
    print("\n[ORIGINAL — 300 karakter pertama]")
    print(df['content'].iloc[i][:300] + "...")
    print("\n[DATA CLEANING]")
    print(df['data_cleaning'].iloc[i][:300] + "...")
    print("\n[CASE FOLDING]")
    print(df['case_folding'].iloc[i][:300] + "...")
    print("\n[NORMALIZATION — hasil akhir]")
    print(df['normalization'].iloc[i][:300] + "...")
    print("-" * 60)


BERITA 1:

[ORIGINAL — 300 karakter pertama]
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,7...

[DATA CLEANING]
Korps Pemberantasan Tindak Pidana Korupsi Kortas Tipikor Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi EBTKE Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya PJUTS Polri menyebut kerugian dalam kasus ini Rp Dirtindak Kortastipidko...

[CASE FOLDING]
korps pemberantasan tindak pidana korupsi kortas tipikor polri mengungkap kasus dugaan korupsi pada ditjen energi baru terbarukan dan konservasi energi ebtke kementerian esdm terkait pengadaan penerangan jalan umum tenaga surya pjuts polri menyebut kerugian dalam kasus ini rp dirtindak kortastipidko...

[NORMALI

In [10]:
bertopic_data = df[[
    'content',          # teks asli
    'data_cleaning',    # setelah hapus URL, artefak web, angka, simbol
    'case_folding',     # setelah huruf kecil
    'normalization',    # hasil akhir siap BERTopic
]].copy()

bertopic_data['preprocessing_stats'] = bertopic_data.apply(
    lambda row: f"orig:{len(row['content'])} -> final:{len(row['normalization'])}",
    axis=1
)

output_file = "cleanData/bertopic_ready_news.csv"
bertopic_data.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data diekspor ke   : {output_file}")
print(f"Jumlah dokumen siap: {len(bertopic_data)}")
print(f"Kolom              : {bertopic_data.columns.tolist()}")

Data diekspor ke   : cleanData/bertopic_ready_news.csv
Jumlah dokumen siap: 234
Kolom              : ['content', 'data_cleaning', 'case_folding', 'normalization', 'preprocessing_stats']
